In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

print("Imports done!")

class ScratchCNN(nn.Module):
    def __init__(self, num_classes=38):
        super(ScratchCNN, self).__init__()

        # 3 convolutional layers
        self.conv1 = nn.Conv2d(in_channels=3,  out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)

        # Max pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Flatten → Linear
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 224 → 112
        x = self.pool(F.relu(self.conv2(x)))   # 112 → 56
        x = self.pool(F.relu(self.conv3(x)))   #  56 → 28

        x = x.view(x.size(0), -1)             # flatten: [batch, 128*28*28]

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model_scratch = ScratchCNN(num_classes=38)
print(model_scratch)

total = sum(p.numel() for p in model_scratch.parameters())
print(f"Scratch CNN total parameters: {total:,}")
print(f"\nFor comparison:")
print(f"EfficientNet-B0 total parameters: 5,288,548")
print(f"EfficientNet trainable (frozen):       48,678")
print(f"\nScratch CNN is training ALL {total:,} params from zero")
print(f"EfficientNet only trained 48,678 params on top of pretrained knowledge")

import os, json

os.makedirs("/root/.kaggle", exist_ok=True)
kaggle_credentials = {
    "username": "your kaggle username",
    "key": "your kaggle key"
}
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
import zipfile
with zipfile.ZipFile("plantvillage-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("plantvillage")
print("Dataset ready!")

Imports done!
ScratchCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=100352, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=38, bias=True)
)
Scratch CNN total parameters: 25,793,382

For comparison:
EfficientNet-B0 total parameters: 5,288,548
EfficientNet trainable (frozen):       48,678

Scratch CNN is training ALL 25,793,382 params from zero
EfficientNet only trained 48,678 params on top of pretrained knowledge
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:17<00:00, 122MB/s]

Dataset ready!


In [2]:
# How much data do we have vs how much the model needs to learn?

scratch_params = 25_225_510  # from day8 parameter count
training_images = 43_444     # 80% of 54,305

print("=== The Scratch CNN problem ===")
print(f"Parameters to learn:  {scratch_params:,}")
print(f"Training images:      {training_images:,}")
print(f"Ratio: {scratch_params // training_images:,} parameters per image")
print()
print("Each image must teach the model ~580 things.")
print("That's like learning a language from 43K sentences")
print("when the language has 25 million words.")
print()
print("=== EfficientNet solution ===")
print(f"Parameters to learn:       48,678")
print(f"Training images:           {training_images:,}")
print(f"Ratio: {48_678 // training_images} parameters per image")
print()
print("EfficientNet already knows edges, textures, shapes from ImageNet.")
print("We only taught it: 'here's what diseased leaves look like'.")

=== The Scratch CNN problem ===
Parameters to learn:  25,225,510
Training images:      43,444
Ratio: 580 parameters per image

Each image must teach the model ~580 things.
That's like learning a language from 43K sentences
when the language has 25 million words.

=== EfficientNet solution ===
Parameters to learn:       48,678
Training images:           43,444
Ratio: 1 parameters per image

EfficientNet already knows edges, textures, shapes from ImageNet.
We only taught it: 'here's what diseased leaves look like'.


In [3]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import torch.nn as nn

# Load pretrained EfficientNet
efficientnet = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

print("=== What EfficientNet learned from ImageNet ===")
print()
print("ImageNet training:")
print("  - 1.2 million images")
print("  - 1,000 different classes")
print("  - Trained for days on powerful hardware")
print()
print("What it learned to detect:")
print("  Layer 1 (conv1): edges, lines, colour gradients")
print("  Layer 2-3:       corners, curves, simple textures")
print("  Layer 4-6:       patterns, spots, stripes")
print("  Layer 7+:        object parts (eyes, leaves, fur, bark)")
print("  Final layer:     full objects (cat, car, plant...)")
print()
print("When we froze all layers except the last:")
print("  → All that knowledge stayed intact")
print("  → We only taught the last layer to map")
print("     'these patterns = Apple_scab'")
print("     'these patterns = Tomato_blight'  etc.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 88.1MB/s]


=== What EfficientNet learned from ImageNet ===

ImageNet training:
  - 1.2 million images
  - 1,000 different classes
  - Trained for days on powerful hardware

What it learned to detect:
  Layer 1 (conv1): edges, lines, colour gradients
  Layer 2-3:       corners, curves, simple textures
  Layer 4-6:       patterns, spots, stripes
  Layer 7+:        object parts (eyes, leaves, fur, bark)
  Final layer:     full objects (cat, car, plant...)

When we froze all layers except the last:
  → All that knowledge stayed intact
  → We only taught the last layer to map
     'these patterns = Apple_scab'
     'these patterns = Tomato_blight'  etc.


In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

base_path = "plantvillage/plantvillage dataset/color"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(base_path, transform=transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

print("Data loaded!")

Data loaded!


In [5]:

print()
print("=== FINAL COMPARISON ===")
print(f"Scratch CNN after 3 epochs:   ~30-50% accuracy (you just saw it)")
print(f"EfficientNet after 3 epochs:  96.62% accuracy (day6)")
print()
print("Same dataset. Same epochs. Same hardware.")
print("The only difference: pretrained knowledge.")


=== FINAL COMPARISON ===
Scratch CNN after 3 epochs:   ~30-50% accuracy (you just saw it)
EfficientNet after 3 epochs:  96.62% accuracy (day6)

Same dataset. Same epochs. Same hardware.
The only difference: pretrained knowledge.
